# Structured Product Pricing

This notebook prices an **Autocallable** structured note via Monte Carlo simulation.

The product is decomposed into two legs:
- **Funding Leg** *(Callable Floating Bond)*: replicates the issuer's funding cost, modelled as a floating-rate bond that is called whenever the autocall trigger is breached.
- **Option Leg** *(Autocallable Phoenix)*: embeds the conditional coupon, memory, autocall, and capital-at-risk (put) features.

The fair-value upfront price is:

$$\text{Price Upfront} = \underbrace{\text{PV Funding Leg}}_{\text{Callable Floating Bond}} - \underbrace{\text{PV Option Leg}}_{\text{Autocallable Phoenix}}$$

In [13]:
import tensorquant as tq
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import date

## Pricing Functions

Helper functions used to evaluate the two legs of the structured note via path-by-path Monte Carlo discounting.

In [14]:
def final_redemption(S_T, strike: float, partecipation: float):
    """
    Payoff of a put as a percentage of notional:

        final_redemption(S_T, K, partecipation)
            = max(K - S_T, 0) / K * partecipation
    """
    return - np.maximum(strike - S_T, 0) / strike * partecipation

In [15]:
def price_funding_bond_from_product(product, s_t, date_grid, disc_curve, daycounter,
                                    forward_rates, forward_pay_dates,
                                    valuation_date):
    """Monte Carlo pricer for a Callable Floating Bond leg tied to an AutocallableOption.

    It prices the floating leg as a callable bond: the forward coupons are
    paid and discounted only while the bond is still alive (until it is called).
    The bond callability replicates the product's autocall feature: at each
    autocall fixing date, if spot >= barrier, the bond terminates and all
    subsequent cashflows are set to zero.

    Args:
        product           : AutocallableOption (barriers and autocall dates).
        s_t               : tensor [n_paths, n_dates] containing spot simulations.
        date_grid         : list of dates mapped to the columns of `s_t`.
        disc_curve        : discount curve.
        daycounter        : day counter.
        forward_rates     : annual forward rates of the floating leg.
        forward_pay_dates : floating-leg payment dates.
        valuation_date    : valuation date.

    Returns:
        PV as a percentage of notional.
    """
    autocall_fixing_dates = product.autocall_fixing_dates
    autocall_barriers     = product.autocall_barrier
    strike                = product.strike

    n_paths      = s_t.shape[0]
    date_to_col  = {d: j for j, d in enumerate(date_grid)}
    autocall_map = {d: i for i, d in enumerate(autocall_fixing_dates)}

    # Cashflow = forward_rate * tau 
    coupon_map = {}
    prev = valuation_date
    for rate, d in zip(forward_rates, sorted(forward_pay_dates)):
        if d > valuation_date:
            coupon_map[d] = rate * daycounter.year_fraction(prev, d)
        prev = d

    # --- Loop Monte Carlo ---
    alive    = np.ones(n_paths, dtype=bool)
    total_pv = np.zeros(n_paths, dtype=np.float64)

    all_dates = sorted(set(autocall_fixing_dates) | set(forward_pay_dates))

    for current_date in all_dates:
        if current_date <= valuation_date:
            continue
        if not alive.any():
            break

        df = float(disc_curve.discount(current_date))

        # Coupon forward: pay if alive
        if current_date in coupon_map:
            total_pv += np.where(alive, coupon_map[current_date] * df, 0.0)

        # Callability
        if current_date in autocall_map:
            a_idx   = autocall_map[current_date]
            barrier = autocall_barriers[a_idx] / 100.0 * strike
            col     = date_to_col[current_date]
            s_i     = s_t[:, col].numpy()
            alive   = alive & (s_i < barrier)

    return float(total_pv.mean())
    
def price_autocallable_from_product(product, s_t, date_grid, disc_curve, valuation_date):
    """Monte Carlo pricer for an AutocallableOption, using the product schedules.

    Loops over all unique observation/fixing dates (union of coupon and autocall), in chronological order.
    For each date, it checks whether it is a coupon fixing date, an autocall fixing date, or both.

    Args:
        product       : an AutocallableOption instance.
        s_t           : tensor [n_paths, len(date_grid)] containing the simulated spots.
        date_grid     : list of dates corresponding to the columns of `s_t`.
        disc_curve    : discount curve providing `.discount(date)`.
        valuation_date: valuation date.

    Returns:
        price as a percentage of notional (e.g. 0.15 = 15%).
    """
    strike             = product.strike
    coupon_fixing_dates  = product.coupon_fixing_dates
    coupon_payment_dates = product.coupon_payment_dates
    coupon_rates         = product.coupon_rates
    coupon_barriers      = product.coupon_barriers
    autocall_fixing_dates  = product.autocall_fixing_dates
    autocall_payment_dates = product.autocall_payment_dates
    autocall_barriers      = product.autocall_barrier
    payoff_barrier       = product.payoff_barrier
    payoff_participation = product.payoff_participation
    memory               = product.memory
    redemption_payoff    = final_redemption 

    n_paths = s_t.shape[0]

    date_to_col = {d: j for j, d in enumerate(date_grid)}

    # lookup: fixing_date → (index, payment_date)
    coupon_map   = {d: (i, coupon_payment_dates[i])
                    for i, d in enumerate(coupon_fixing_dates)}
    autocall_map = {d: (i, autocall_payment_dates[i])
                    for i, d in enumerate(autocall_fixing_dates)}

    all_fixing_dates = sorted(set(coupon_fixing_dates) | set(autocall_fixing_dates))

    alive          = np.ones(n_paths,  dtype=bool)
    unpaid_coupons = np.zeros(n_paths, dtype=np.float64)
    total_pv       = np.zeros(n_paths, dtype=np.float64)

    for fix_date in all_fixing_dates:

        if fix_date <= valuation_date:
            continue
        if not alive.any():
            break

        col = date_to_col[fix_date]
        s_i = s_t[:, col].numpy()   

        pay = np.zeros(n_paths, dtype=np.float64)

        # --- coupon leg ---
        if fix_date in coupon_map:
            c_idx, c_pay_date = coupon_map[fix_date]
            coupon_thr   = coupon_barriers[c_idx] / 100.0 * strike
            above_coupon = s_i >= coupon_thr
            paid         = alive & above_coupon

            if memory:
                pay_amount = coupon_rates[c_idx] / 100.0 * (1.0 + unpaid_coupons)
            else:
                pay_amount = coupon_rates[c_idx] / 100.0

            pay += np.where(paid, pay_amount, 0.0)

            # final redemption
            if fix_date == coupon_fixing_dates[-1] and redemption_payoff is not None:
                below_barrier = s_i <= payoff_barrier / 100.0 * strike
                pay += np.where(alive & below_barrier,
                                redemption_payoff(s_i, strike, payoff_participation),
                                0.0)

            # discount
            df = float(disc_curve.discount(c_pay_date))
            total_pv += pay * df

            # memory counter
            unpaid_coupons = np.where(paid,                    0.0,                  unpaid_coupons)
            unpaid_coupons = np.where(alive & ~above_coupon,   unpaid_coupons + 1.0, unpaid_coupons)

        # --- autocall leg 
        if fix_date in autocall_map:
            a_idx, _ = autocall_map[fix_date]
            autocall_thr   = autocall_barriers[a_idx] / 100.0 * strike
            above_autocall = s_i >= autocall_thr
            alive          = alive & ~above_autocall

    return float(total_pv.mean())


## Market Setup

Flat rate curves and constant Black volatility are used as market inputs. In production, a full term structure and a local/stochastic volatility model would be calibrated.

In [16]:
ref_date = tq.Settings.evaluation_date

calendar = tq.TARGET()
daycounter = tq.DayCounter(tq.DayCounterConvention.Actual360)
evaluation_date = tq.Settings.evaluation_date

disc_curve = tq.FlatCurve(ref_date, 0.021, tq.DayCounterConvention.Actual360)
eur6m_curve = tq.FlatCurve(ref_date, 0.026, tq.DayCounterConvention.Actual360)
vol = tq.BlackConstantVolatility(ref_date, volatility=0.25)

market = {
    "IR:EUR:ESTR:SPOT":    disc_curve,
    "IR:EUR:6M:SPOT":    eur6m_curve,
    "EQ:EUR:DEFAULT:VOL":  vol
}

market_env = tq.MarketEnvironment(market)

## Structured Product Example

Definition of the Autocallable Certificate and the relevant market parameters.

### Autocallable — Product Description

An **Autocallable** is a capital-at-risk structured note. Let $S_t$ denote the underlying spot, $K_0$ the **initial fixing** (certificate strike, i.e. $S_{t_0}$ at trade inception), and let the note have observation dates $t_1 < t_2 < \cdots < t_n = T$.

---

#### Coupon Dates  $t_i$, $i = 1, \ldots, n$

**Conditional coupon** — paid on each fixing date if the note is still alive and the underlying closes above the coupon barrier $B_c^{(i)}$:

$$
\text{Coupon}_i = c_i \cdot \bigl(1 + m_i\bigr) \cdot \mathbf{1}_{\left\{S_{t_i} \;\geq\; B_c^{(i)} \cdot K_0\right\}}
$$

where $m_i$ is the count of accumulated unpaid coupons (memory feature; $m_i = 0$ when `memory = False`).

**Autocall trigger** — checked on autocall fixing dates; if breached the note terminates immediately and the full notional is redeemed at par:

$$
\text{Autocall at } t_i: \quad S_{t_i} \geq B_a^{(i)} \cdot K_0 \;\Longrightarrow\; \text{early redemption at } 100\%
$$

---

#### Final Redemption Date $T$ (if not called)

At maturity the investor receives par **plus** the following capital-at-risk payoff. The embedded put has its own strike $K_{\text{put}}$, which in general may differ from the initial fixing $K_0$ (in this example both equal 100):

$$
\text{Capital P\&L}_T =
\begin{cases}
0 & \text{if } S_T > B_{\text{put}} \cdot K_0 \\[6pt]
-\;p \cdot \dfrac{K_{\text{put}} - S_T}{K_{\text{put}}} & \text{if } S_T \leq B_{\text{put}} \cdot K_0
\end{cases}
$$

Two distinct levels govern this payoff:

| Level | Symbol | Role |
|---|---|---|
| **Put barrier** | $B_{\text{put}} \cdot K_0$ | *Activation threshold.* The capital loss is triggered only if $S_T$ closes **at or below** this level. Above it the investor is fully protected. |
| **Put strike** | $K_{\text{put}}$ | *Loss reference.* Once the barrier is breached, the loss is measured from $K_{\text{put}}$, **not** from the barrier level. |

Because $B_{\text{put}} < 1$, the put strike lies strictly above the barrier ($K_{\text{put}} > B_{\text{put}} \cdot K_0$ when $K_{\text{put}} = K_0$). This creates a **discontinuity**: the moment $S_T$ breaches the barrier from above, the loss jumps immediately to $p \cdot (1 - B_{\text{put}})$, then grows linearly as the underlying falls further.

**Example** — with $B_{\text{put}} = 80\%$, $K_{\text{put}} = 100$, $p = 100\%$ and $S_T = 70$:

$$
\text{Capital P\&L}_T = -\,p \cdot \frac{K_{\text{put}} - S_T}{K_{\text{put}}} = -1 \cdot \frac{100 - 70}{100} = -30\%
$$

The investor receives $100\% - 30\% = 70\%$ of notional, participating fully in the downside below the barrier.

The net redemption seen by the investor is therefore $100\% + \text{Capital P\&L}_T$.

---

| Parameter | Symbol | Value (this example) |
|---|---|---|
| Initial fixing (certificate strike) | $K_0$ | 100 |
| Put strike | $K_{\text{put}}$ | 100 |
| Coupon barrier | $B_c$ | 80 % |
| Coupon rate | $c_i$ | 4 % per period |
| Autocall barrier | $B_a$ | 100 % |
| Put barrier | $B_{\text{put}}$ | 80 % |
| Participation | $p$ | 100 % |
| Memory | — | Off |

In [17]:
currency = tq.Currency.EUR
notional = 100e6
start_date = ref_date
end_date = calendar.advance(ref_date, 5, tq.TimeUnit.Years, tq.BusinessDayConvention.ModifiedFollowing)

spot   = 100
strike = 100

schedule_gen = tq.ScheduleGenerator(calendar, tq.BusinessDayConvention.ModifiedFollowing)
coupon_fixing_dates = schedule_gen.generate(ref_date, end_date, 6, tq.TimeUnit.Months)[1:]
coupon_payment_dates = coupon_fixing_dates
coupon_rates = [4.] *len(coupon_fixing_dates)
coupon_barriers = [80]*len(coupon_fixing_dates)
memory = False

first_autocall_date = calendar.advance(ref_date, 1, tq.TimeUnit.Years, tq.BusinessDayConvention.ModifiedFollowing) #coupon_fixing_dates[0]
autocall_fixing_dates = schedule_gen.generate(first_autocall_date, end_date, 6, tq.TimeUnit.Months)
autocall_payment_dates = autocall_fixing_dates
autocall_barrier = [100]*len(autocall_fixing_dates) 

payoff_barrier = 80.
payoff_partecipation = 1.
payoff_type = 'put'


option_leg = tq.AutocallableOption(
            ccy=currency,
            notional=notional,
            start_date=start_date,
            end_date=end_date,
            strike=strike,
            coupon_fixing_dates=coupon_fixing_dates,
            coupon_payment_dates=coupon_payment_dates,
            coupon_rates=coupon_rates,
            coupon_barriers=coupon_barriers,
            memory=memory,
            autocall_fixing_dates=autocall_fixing_dates,
            autocall_payment_dates=autocall_payment_dates,
            autocall_barrier=autocall_barrier,
            payoff_barrier=payoff_barrier,
            payoff_participation=payoff_partecipation,
            payoff_type=payoff_type,
        )

### Monte Carlo Scenario Simulations

Underlying equity paths are simulated using a **Geometric Brownian Motion** on a monthly discretisation grid. The same path set is shared across both the funding leg and the option leg pricers.

In [18]:
model  = tq.GeometricBrownianMotion(mu=disc_curve.rate, sigma=vol.volatility().numpy(), x0=spot)

coupon_dates   = option_leg.coupon_fixing_dates
autoc_dates = option_leg.autocall_fixing_dates
discretization_grid = schedule_gen.generate(ref_date, end_date, 1, tq.TimeUnit.Months)
date_grid = sorted(set(coupon_dates) | set(autoc_dates) | set(discretization_grid))
time_grid = [daycounter.year_fraction(ref_date, d) for d in date_grid]

n_path = 20000
z   = tf.random.normal((n_path, len(time_grid)), seed=12, dtype=tf.dtypes.float64)
s_t = model.evolve(time_grid, z)

## Funding Leg — Callable Floating Bond

The funding leg is modelled as a **callable floating-rate bond**: the issuer pays EURIBOR 6M coupons until the earlier of maturity or the first autocall event. Its present value is computed via Monte Carlo, conditional on the same autocall triggers as the option leg.

In [19]:
currency = tq.Currency.EUR
mod_fol_convention = tq.BusinessDayConvention.ModifiedFollowing
eur6m_index = tq.IborIndex(calendar, 6, tq.TimeUnit.Months, tq.Currency.EUR, 2)

settlement_delay = 3
period_fixed_leg = "1Y"
period_float_leg = "6M"

irs_eur6m_generator = tq.SwapGenerator(currency, settlement_delay,
                                        period_fixed_leg, period_float_leg,
                                        mod_fol_convention, notional,
                                        tq.DayCounterConvention.Actual360, tq.DayCounterConvention.Actual360,
                                        calendar, eur6m_index)
swap = irs_eur6m_generator.build(trade_date=ref_date, quote=0.0, term="5Y")

In [20]:
def price_funding_bond_from_product(product, s_t, date_grid, disc_curve, daycounter,
                                    forward_rates, forward_pay_dates,
                                    valuation_date):
    """Monte Carlo pricer for a Callable Floating Bond leg tied to an AutocallableOption.

    It prices the floating leg as a callable bond: the forward coupons are
    paid and discounted only while the bond is still alive (until it is called).
    The bond callability replicates the product's autocall feature: at each
    autocall fixing date, if spot >= barrier, the bond terminates and all
    subsequent cashflows are set to zero.

    Args:
        product           : AutocallableOption (barriers and autocall dates).
        s_t               : tensor [n_paths, n_dates] containing spot simulations.
        date_grid         : list of dates mapped to the columns of `s_t`.
        disc_curve        : discount curve.
        daycounter        : day counter.
        forward_rates     : annual forward rates of the floating leg.
        forward_pay_dates : floating-leg payment dates.
        valuation_date    : valuation date.

    Returns:
        PV as a percentage of notional.
    """
    autocall_fixing_dates = product.autocall_fixing_dates
    autocall_barriers     = product.autocall_barrier
    strike                = product.strike

    n_paths      = s_t.shape[0]
    date_to_col  = {d: j for j, d in enumerate(date_grid)}
    autocall_map = {d: i for i, d in enumerate(autocall_fixing_dates)}

    # Cashflow = forward_rate * tau per ogni periodo, pre-calcolati
    coupon_map = {}
    prev = valuation_date
    for rate, d in zip(forward_rates, sorted(forward_pay_dates)):
        if d > valuation_date:
            coupon_map[d] = rate * daycounter.year_fraction(prev, d)
        prev = d

    # --- Loop Monte Carlo ---
    alive    = np.ones(n_paths, dtype=bool)
    total_pv = np.zeros(n_paths, dtype=np.float64)

    all_dates = sorted(set(autocall_fixing_dates) | set(forward_pay_dates))

    for current_date in all_dates:
        if current_date <= valuation_date:
            continue
        if not alive.any():
            break

        df = float(disc_curve.discount(current_date))

        # Coupon forward: paga solo se vivo
        if current_date in coupon_map:
            total_pv += np.where(alive, coupon_map[current_date] * df, 0.0)

        # Callability: i path sopra la barriera escono
        if current_date in autocall_map:
            a_idx   = autocall_map[current_date]
            barrier = autocall_barriers[a_idx] / 100.0 * strike
            col     = date_to_col[current_date]
            s_i     = s_t[:, col].numpy()
            alive   = alive & (s_i < barrier)

    return float(total_pv.mean())

In [21]:
swap_engine = tq.SwapPricer()
swap_engine.price(swap, market_env, True)
forward_rates = swap.floating_leg.display_flows()['rate'].to_list()
forward_pay_dates = swap.floating_leg.display_flows()['pay_date'].to_list()


In [26]:
funding_bond_pv = price_funding_bond_from_product(
    product=option_leg,
    s_t=s_t,
    date_grid=date_grid,
    disc_curve=disc_curve,
    daycounter=daycounter,
    forward_rates=forward_rates,
    forward_pay_dates=forward_pay_dates,
    valuation_date=ref_date,
)

print(f"Callable Bond PV  (% notional): {funding_bond_pv*100:,.2f}")
# print(f"Plain Bond PV     (% notional): {swap.floating_leg.price / notional * 100:,.2f}")

Callable Bond PV  (% notional): 5.06


## Option Leg

Present value of the structured option leg, including conditional coupons, memory effect, autocall termination, and the capital-at-risk put payoff at maturity.

In [23]:
product = tq.AutocallableOption(
    ccy=currency,
    notional=notional,
    start_date=start_date,
    end_date=end_date,
    strike=strike,
    coupon_fixing_dates=coupon_fixing_dates,
    coupon_payment_dates=coupon_payment_dates,
    coupon_rates=coupon_rates,
    coupon_barriers=coupon_barriers,
    memory=memory,
    autocall_fixing_dates=autocall_fixing_dates,
    autocall_payment_dates=autocall_payment_dates,
    autocall_barrier=autocall_barrier,
    payoff_barrier=payoff_barrier,
    payoff_participation=payoff_partecipation,
    payoff_type=payoff_type,
)

In [24]:
p_opt = price_autocallable_from_product(product, s_t, date_grid, disc_curve, ref_date)

print(f"Option Leg PV  (% notional): {p_opt*100:,.2f}")
print(f"Price Upfront (% notional): {(funding_bond_pv - p_opt)*100:,.2f}")

Option Leg PV  (% notional): 1.25
Price Upfront (% notional): 3.81
